# Pelagic spin-up validation notebook

This notebook follows the following workflow:

1. **Configuration** for paths, variables, layers, and output folders.
2. **Helper functions** for loading model output and handling grid/time quirks.
3. **Analysis functions** for trends, subdomain diagnostics, observations, and satellite comparison.
4. **Entry-point examples** that can be toggled on when you want to run a specific workflow.

The main cleanup change is that map-based diagnostics now **export individual PNG frames** instead of rendering animations inline in the notebook.


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib import animation

from IPython.display import HTML

import numpy as np
import pandas as pd
import seaborn as sns
import xarray as xr

import os
import cmocean

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import contextily as ctx

import glob
from PIL import Image
import imageio.v3 as iio
import scienceplots

In [ ]:

# ------------------------------ Helpers -----------------------------------
def _find_time_dim(da: xr.DataArray) -> str:
    for dim in da.dims:
        if "time" in dim.lower():
            return dim
    raise ValueError(f"No time-like dimension found in {da.dims}")

def _find_vertical_dim(da: xr.DataArray, time_dim: str) -> str:
    candidates = ("z", "sigma", "layer", "level", "lev", "depth", "nmesh2_layer_3d")
    for dim in da.dims:
        if dim == time_dim:
            continue
        if any(key in dim.lower() for key in candidates):
            return dim
    # Fallback: assume second dimension is vertical for 4D fields [time, z, y, x]
    if len(da.dims) >= 3:
        return da.dims[1]
    raise ValueError(f"No vertical-like dimension found in {da.dims}")

def _pick_coord_name(ds: xr.Dataset, candidates: tuple[str, ...]) -> str | None:
    for name in candidates:
        if name in ds.variables:
            return name
    return None

def _pick_existing_var(ds: xr.Dataset, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in ds.variables:
            return name
    raise KeyError(f"Could not find any of these variables: {candidates}")

def _find_horizontal_dims(da: xr.DataArray, exclude: tuple[str, ...] = ()) -> tuple[str, str]:
    dims = [dim for dim in da.dims if dim not in exclude]
    if len(dims) != 2:
        raise ValueError(f"Expected 2 horizontal dims, got {dims} from {da.dims}")
    return dims[0], dims[1]

def _compute_edges(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    if values.size < 2:
        raise ValueError("Need at least two points to build cell edges.")
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (values[:-1] + values[1:])
    edges[0]    = values[0]  - 0.5 * (values[1]  - values[0])
    edges[-1]   = values[-1] + 0.5 * (values[-1] - values[-2])
    return edges

def _make_vertical_edges(surface_values: np.ndarray, bottom_values: np.ndarray) -> np.ndarray:
    bottom_edges       = _compute_edges(bottom_values)
    surface_edges      = _compute_edges(surface_values)
    water_column_height = surface_edges - bottom_edges
    return bottom_edges[None, :] + sigma_edges * water_column_height[None, :]

def _drop_duplicate_time(da: xr.DataArray, time_dim: str) -> xr.DataArray:
    """Remove duplicate timestamps along the time dimension."""
    time_values = da[time_dim].values
    _, unique_idx = np.unique(time_values, return_index=True)
    unique_idx = np.sort(unique_idx)
    return da.isel({time_dim: unique_idx})


def export_mp4_from_frames(frame_dir,
                           output_path,
                           fps=40): 

    frame_dir = Path(frame_dir)
    output_path = Path(output_path)

    frame_files = sorted(frame_dir.rglob("*.png"))
    if not frame_files:
        print(f"No PNG frames found in {frame_dir}")
        return

    print(f"Creating MP4 animation → {output_path}")
    print(f"Found {len(frame_files)} frames")

    writer = animation.FFMpegWriter(
        fps=fps,
        codec="libx264",
        bitrate=1800,
        extra_args=["-pix_fmt", "yuv420p"]   # <-- Windows compatible
    )

    first_img = plt.imread(frame_files[0])
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(first_img)
    ax.axis("off")

    with writer.saving(fig, str(output_path), dpi=200):
        for f in frame_files:
            img = plt.imread(f)
            im.set_data(img)
            writer.grab_frame()

    plt.close(fig)
    print(f"MP4 saved: {output_path}")

def extract_transect(
    ds: xr.Dataset,
    var_name: str,
    i_lon: int,
    lat_start: int,
    lat_stop: int,
    lon_name_candidates: tuple[str, ...] = ("lonc", "lon", "longitude"),
    lat_name_candidates: tuple[str, ...] = ("latc", "lat", "latitude"),
) -> dict[str, xr.DataArray]:
    da = ds[var_name]
    time_dim = _find_time_dim(da)
    z_dim = _find_vertical_dim(da, time_dim)
    y_dim, x_dim = _find_xy_dims(da, time_dim, z_dim)

    transect = da.isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})

    lat_name = _pick_coord_name(ds, lat_name_candidates)
    if lat_name is None:
        raise KeyError(f"Could not find latitude variable among {lat_name_candidates}")

    lat_da = ds[lat_name]
    if y_dim in lat_da.dims and x_dim in lat_da.dims:
        lat_line = lat_da.isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})
    elif y_dim in lat_da.dims:
        lat_line = lat_da.isel({y_dim: slice(lat_start, lat_stop)})
    else:
        raise ValueError(f"Latitude variable '{lat_name}' does not match transect dims")

    if "h" in ds.variables:
        h = ds["h"].isel({x_dim: i_lon, y_dim: slice(lat_start, lat_stop)})
        if z_dim in h.dims:
            depth = -h.cumsum(dim=z_dim)
        else:
            depth = None
    else:
        depth = None

    return {
        "transect": transect,
        "lat_line": lat_line,
        "depth": depth,
        "time_dim": transect.dims[0],
        "z_dim": z_dim,
        "y_dim": y_dim,
    }


In [ ]:
# Generate static image for testing
# --- Configuration ---
base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
out_dir = "/export/lv9/projects/dws/results/animation/pelagic"
spinup = "spinup_06"
date_str = "201508"

variable = "P1c" 
label = "Diatoms (P1c)"

#variable = "P2c" 
#label = "Flagellates (P2c)"

#variable = "P3c" 
#label = "PicoPhytoPlankton (P3c)"

# Fixed map window: Dutch Wadden Sea to Ems-Dollard
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# --- Load Data ---
file_path = os.path.join(base_dir, spinup, f"dws_500m.3d.{date_str}.nc")
ds = xr.open_dataset(file_path)

# Extract variable. If the variable is 3D (e.g., pelagic), we safely isolate the bottom layer.
data_var = ds[variable]
if 'level' in data_var.dims:
    data_var = data_var.isel(level=-1) 

# Select the first time step in the daily output file
data_slice = data_var.isel(time=0)

# To estimate the range of the measurements
VMIN = np.nanpercentile(data_slice.values, 1)
VMAX = np.nanpercentile(data_slice.values, 99)

VMIN = np.nanmin(data_slice.values)
VMAX = np.nanmax(data_slice.values)

# Fixed color scale configuration to prevent map flickering
VMIN = 0
VMAX = 4000
print(VMIN, VMAX)

CMAP = cmocean.cm.matter
#CMAP = cmocean.cm.thermal

i_lon = 152 # J value for Griend island
lat_start = 90
lat_stop = 153

# To log the values (optional)
log_norm = mcolors.LogNorm(vmin=VMIN, vmax=VMAX)

# Extract 2D coordinate arrays and mathematically fill any spatial gaps (NaNs) 
# usually found in the masked land boundaries of the domain mesh.
lon_da = ds.lonc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')
lat_da = ds.latc.ffill(dim='xc').bfill(dim='xc').ffill(dim='yc').bfill(dim='yc')

lon = lon_da.values
lat = lat_da.values

# Extract transect coordinates from the curvilinear grid
transect_lon = lon[lat_start:lat_stop, i_lon]
transect_lat = lat[lat_start:lat_stop, i_lon]

# Remove NaNs
valid = np.isfinite(transect_lon) & np.isfinite(transect_lat)

transect_lon = transect_lon[valid]
transect_lat = transect_lat[valid]


# --- Plotting ---
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.Mercator())
ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

## Add map features
#ax.add_feature(cfeature.LAND, zorder=2, edgecolor='k', facecolor='lightgray')
#ax.add_feature(cfeature.COASTLINE, zorder=2)

# Add the high-resolution background
ctx.add_basemap(ax, 
                crs=ax.projection.to_string(), 
                #source=ctx.providers.CartoDB.Positron,
                source=ctx.providers.Esri.WorldShadedRelief,
                zorder=1,
               )

txt = ax.texts[-1]
txt.set_position([0.99,0.02])
txt.set_ha('right')

# Plot spatial distribution
mesh = ax.pcolormesh(lon, lat, data_slice.values,
                     transform=ccrs.PlateCarree(),
                     cmap=CMAP,
                     #norm=log_norm,
                     vmin=VMIN,
                    vmax=VMAX,
                     shading='gouraud',
                     zorder=2)

ax.plot(
    transect_lon,
    transect_lat,
    linestyle="--",
    color="black",
    linewidth=2,
    transform=ccrs.PlateCarree(),
    zorder=5,
)

# Formatting
cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
cbar.ax.tick_params(labelsize=10)

# Dynamically fetch units if available in the netCDF metadata
#units = ds[variable].attrs.get('units', '')
cbar.set_label(r"Biomass (mgC m$^{-3}$)", fontsize=12)

# Extract human-readable time
time_str = str(data_slice.time.values).split('T')[0]
plt.title(f"{label} | {spinup} | {time_str}", fontsize=12, pad=10)

plt


In [ ]:
# -----------------------------------------------------------------------------
# CONFIGURATION
# -----------------------------------------------------------------------------

base_dir = "/export/lv9/projects/dws/model_output/archived_runs"
frames_root = "/export/lv9/projects/dws/results/animation/pelagic"
os.makedirs(frames_root, exist_ok=True)

variable    = "salt"         # <--- your requested variable
label       = "Salinity (psu)"

# variable    = "temp"         
# label       = "Temperature (degrees)"

# variable = "P1c" 
# label = "Diatoms (P1c)"

#variable = "P2c" 
#label = "Flagellates (P2c)"

#variable = "P3c" 
#label = "PicoPhytoPlankton (P3c)"

# Color scale
VMIN = 0
VMAX = 42
CMAP = cmocean.cm.haline

#VMIN = 0
#VMAX = 22
#CMAP = cmocean.cm.thermal

# Diatom
#VMIN = 0
#VMAX = 4000
#CMAP = cmocean.cm.matter

YEAR = 2015

TOPVIEW_DIR  = Path(frames_root) / variable / "topview"
TRANSECT_DIR = Path(frames_root) / variable / "transect"

TOPVIEW_DIR.mkdir(parents=True, exist_ok=True)
TRANSECT_DIR.mkdir(parents=True, exist_ok=True)

# Map window
MAP_EXTENT_LONLAT = (4.0, 6.6, 52.5, 53.8)

# Transect definition
I_LON = 152
LAT_START = 90
LAT_STOP = 153




In [ ]:
# topview frames

spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])
frame_counter = 0

for spinup in spinups:
    print(f"\n=== Processing spinup: {spinup} ===")

    spinup_dir = os.path.join(base_dir, spinup)
    nc_files = sorted(glob.glob(os.path.join(spinup_dir, "dws_500m.3d.*.nc")))

    for file_path in nc_files:
        print(f"  → Reading {file_path}")

        with xr.open_dataset(file_path) as ds:

            # Fix curvilinear grid
            lon_da = ds.lonc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lat_da = ds.latc.ffill("xc").bfill("xc").ffill("yc").bfill("yc")
            lon = lon_da.values
            lat = lat_da.values

            # Extract variable
            data_var = ds[variable]
            if "level" in data_var.dims:
                data_var = data_var.isel(level=-1)

            # Mask variable
            mask_da = ds["elev"].squeeze(drop=True)
            mask_time_dim = _find_time_dim(mask_da)
            mask_da = _drop_duplicate_time(mask_da, mask_time_dim)

            # Build transect
            tran_lon = lon[LAT_START:LAT_STOP, I_LON]
            tran_lat = lat[LAT_START:LAT_STOP, I_LON]
            valid = np.isfinite(tran_lon) & np.isfinite(tran_lat)
            tran_lon = tran_lon[valid]
            tran_lat = tran_lat[valid]

            # Loop through time steps
            for t_idx in range(data_var.sizes["time"]):

                frame = data_var.isel(time=t_idx).values.astype(float)

                # Apply mask
                mask2d = mask_da.isel({mask_time_dim: t_idx}).values
                wet_mask = mask2d > -1.1869
                frame[~wet_mask] = np.nan

                timestamp = str(ds.time.values[t_idx])[:10]  # YYYY-MM-DD

                # Plot
                fig = plt.figure(figsize=(8, 6))
                ax = plt.axes(projection=ccrs.Mercator())
                ax.set_extent(MAP_EXTENT_LONLAT, crs=ccrs.PlateCarree())

                # Basemap
                ctx.add_basemap(
                    ax,
                    crs=ax.projection.to_string(),
                    source=ctx.providers.Esri.WorldShadedRelief,
                    zorder=1,
                )
                txt = ax.texts[-1]
                txt.set_position([0.99, 0.02])
                txt.set_ha("right")

                # Field
                mesh = ax.pcolormesh(
                    lon,
                    lat,
                    frame,
                    transform=ccrs.PlateCarree(),
                    cmap=CMAP,
                    vmin=VMIN,
                    vmax=VMAX,
                    shading="gouraud",
                    zorder=2,
                )

                # Transect
                ax.plot(
                    tran_lon,
                    tran_lat,
                    "--",
                    color="black",
                    linewidth=2,
                    transform=ccrs.PlateCarree(),
                    zorder=5,
                )

                # Colorbar
                cbar = plt.colorbar(mesh, ax=ax, shrink=0.9, pad=0.05)
                cbar.set_label("Biomass (mgC m$^{-3}$)", fontsize=12)

                ax.set_title(f"{label} | {spinup} | {timestamp}")

                # Save frame with:
                #   - global counter (for MP4)
                #   - date (for readability)
                frame_path = os.path.join(
                    TOPVIEW_DIR,
                    f"{variable}_{frame_counter:06d}_{timestamp}.png"
                )

                plt.savefig(frame_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

                frame_counter += 1

print("\nDone. Frames saved in:")
print(TOPVIEW_DIR)

In [ ]:
# export the transect frames to MP4 to topview directory
export_mp4_from_frames(
    TOPVIEW_DIR,
    output_path= Path(frames_root) / variable / f"{variable}_topview.mp4",
    fps=40
)

In [ ]:
# -----------------------------
# USER SETTINGS FOR TRANSECT FRAMES
# -----------------------------


INDEX_LON_TRANSECT = 152
LAT_START = 90
LAT_STOP  = 153

ELEVATION_VAR_CANDIDATES = ("elev", "eta", "eta_out", "elevation", "ssh", "zeta")
SURFACE_LAYER_INDEX = 10  # 10 for the top layer, and 0 for the bottom layer
BATHY_VAR_CANDIDATES = ("bathymetry",)

# -----------------------------
# PATHS
# -----------------------------
spinups = sorted([d for d in os.listdir(base_dir) if d.startswith("spinup_")])

frame_counter = 0

# helper
def _find_xy_dims(da: xr.DataArray, time_dim: str, z_dim: str) -> tuple[str, str]:
    remain = [d for d in da.dims if d not in (time_dim, z_dim)]
    if len(remain) != 2:
        raise ValueError(
            f"Expected 2 horizontal dims after removing time/z, got {remain} from {da.dims}"
        )
    return remain[0], remain[1]
    
# -----------------------------


In [ ]:
# transect frames

frame_counter = 0   # global counter for MP4 ordering

for spinup in spinups:
    print(f"\n=== Processing transect for {spinup} ===")

    spinup_dir = Path(base_dir) / spinup
    nc_files = sorted(spinup_dir.glob("dws_500m.3d.*.nc"))

    for file_path in nc_files:
        print(f"  → Reading {file_path}")

        with xr.open_dataset(file_path, decode_times=True) as ds:

            # -------------------------------
            # Extract transect
            # -------------------------------
            pack = extract_transect(
                ds=ds,
                var_name=variable,
                i_lon=INDEX_LON_TRANSECT,
                lat_start=LAT_START,
                lat_stop=LAT_STOP,
            )

            transect = pack["transect"]
            lat_line = np.asarray(pack["lat_line"].values, dtype=float)
            time_dim = pack["time_dim"]
            z_dim = pack["z_dim"]
            y_dim = pack["y_dim"]

            # -------------------------------
            # Elevation + bathymetry
            # -------------------------------
            elev_name = _pick_existing_var(ds, ELEVATION_VAR_CANDIDATES)
            bathy_name = _pick_existing_var(ds, BATHY_VAR_CANDIDATES)

            elev_da = ds[elev_name]
            elev_time_dim = _find_time_dim(elev_da)
            elev_y_dim, elev_x_dim = _find_horizontal_dims(elev_da, exclude=(elev_time_dim,))
            elev_line = elev_da.isel({
                elev_x_dim: INDEX_LON_TRANSECT,
                elev_y_dim: slice(LAT_START, LAT_STOP),
            }).transpose(elev_time_dim, elev_y_dim)

            bathy_da = ds[bathy_name]
            bathy_y_dim, bathy_x_dim = _find_horizontal_dims(bathy_da)
            bathy_line = bathy_da.isel({
                bathy_x_dim: INDEX_LON_TRANSECT,
                bathy_y_dim: slice(LAT_START, LAT_STOP),
            })

            # -------------------------------
            # Build static mask
            # -------------------------------
            bottom_vals = np.where(
                np.asarray(bathy_line.values) <= -9990,
                np.nan,
                -np.asarray(bathy_line.values, dtype=float),
            )

            valid_idx = np.flatnonzero(
                np.isfinite(lat_line) & np.isfinite(bottom_vals)
            )
            if valid_idx.size < 2:
                raise ValueError("Not enough valid transect columns.")

            # Apply static mask
            transect = transect.isel({y_dim: valid_idx})
            elev_line = elev_line.isel({elev_y_dim: valid_idx})
            lat_vals = lat_line[valid_idx]
            bottom_vals = bottom_vals[valid_idx]

            # -------------------------------
            # Vertical edges
            # -------------------------------
            nz = transect.sizes[z_dim]
            sigma_edges = np.linspace(0, 1, nz + 1)[:, None]

            ymin = float(np.nanmin(bottom_vals)) - 0.5
            ymax = 1.4

            # -------------------------------
            # Loop through time frames
            # -------------------------------
            times = transect[time_dim]
            frame_indices = np.arange(0, transect.sizes[time_dim], 1)

            for idx in frame_indices:

                temp_vals = np.asarray(
                    transect.isel({time_dim: idx})
                    .transpose(z_dim, y_dim)
                    .values,
                    dtype=float,
                )
                surface_vals = np.asarray(
                    elev_line.isel({elev_time_dim: idx}).values,
                    dtype=float,
                )

                # Wet mask
                valid_frame = (
                    np.isfinite(surface_vals)
                    & np.isfinite(bottom_vals)
                    & (surface_vals > bottom_vals)
                )

                # -------------------------------
                # Plot
                # -------------------------------
                fig, ax = plt.subplots(figsize=(10, 5))

                if valid_frame.sum() >= 2:
                    lat_plot = lat_vals[valid_frame]
                    bottom_plot = bottom_vals[valid_frame]
                    surface_plot = surface_vals[valid_frame]
                    temp_plot = np.ma.masked_invalid(temp_vals[:, valid_frame])

                    x_edges = _compute_edges(lat_plot)
                    x_edges_2d = np.broadcast_to(
                        x_edges[None, :], (temp_plot.shape[0] + 1, x_edges.size)
                    )
                    z_edges_2d = _make_vertical_edges(surface_plot, bottom_plot)

                    ax.pcolormesh(
                        x_edges_2d,
                        z_edges_2d,
                        temp_plot,
                        shading="auto",
                        cmap=CMAP,
                        vmin=VMIN,
                        vmax=VMAX,
                    )

                # Seafloor
                ax.fill_between(
                    lat_vals,
                    bottom_vals,
                    ymin,
                    color="#dcd5c1",
                    alpha=1,
                )

                # Surface line
                if valid_frame.sum() >= 2:
                    ax.plot(lat_plot, surface_plot, color="white", linewidth=1.2)

                # Bathymetry line
                ax.plot(lat_vals, bottom_vals, color="black", linewidth=1.8)

                timestamp = str(times.values[idx])[:10]
                ax.set_title(f"{label} transect | {spinup} | {timestamp}")
                ax.set_xlabel("Latitude")
                ax.set_ylabel("Depth (m)")
                ax.set_ylim(ymin, ymax)
                ax.grid(True, alpha=0.25)

                # Save frame with:
                #   - global counter (for MP4)
                #   - date (for readability)
                frame_path = os.path.join(
                    TRANSECT_DIR,
                    f"{variable}_{frame_counter:06d}_{timestamp}.png"
                )
                fig.savefig(frame_path, dpi=200, bbox_inches="tight")
                plt.close(fig)

                frame_counter += 1

print("\nDone. Frames saved in:")
print(TRANSECT_DIR)

In [ ]:
# export the transect frames to MP4 to transect directory
export_mp4_from_frames(
    TRANSECT_DIR,
    output_path= Path(frames_root) / variable / f"{variable}_transect.mp4",
    fps=40
)
